In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, r2_score, max_error, mean_absolute_percentage_error, mean_absolute_error

from scipy.optimize import curve_fit

import random

In [2]:
df = pd.read_excel('./././data/waste_water/all_actual_data.xlsx')

In [3]:
seconds = np.array(df['min'])*60
df['hours'] = df['min']/60
REAL = np.array(df.iloc[1511, 2:-1]).transpose()
df.head()

,min,day,A0('11-25'),A0('9-16' ),A0('9-10'),A0('9-20'),A0('9-24'),A0('9-27'),A0('9-11'),A0('11-28'),...,D('10-28'),D('10-21'),D('11-13'),D('10-24'),D('10-30'),A('9-30'),B('9-30'),C('9-30'),D('9-30'),hours
0,0.0,0.000000,0.0073,0.0000,0.0061,0.0032,0.0107,0.0136,0.0142,0.0061,...,0.0001,0.0014,0.0020,0.0007,0.0002,0.0286,0.0154,0.0086,0.0209,0.000000
1,5.0,0.003472,0.0394,0.0709,0.0035,0.0029,0.0432,0.0332,0.0124,0.0021,...,0.0007,0.0005,0.0028,0.0008,0.0252,0.0327,0.0229,0.0161,0.0195,0.083333
2,10.0,0.006944,0.0744,0.1349,0.0092,0.0099,0.0819,0.0513,0.0044,0.0036,...,0.0000,0.0006,0.0070,0.0013,0.0039,0.0352,0.0271,0.0211,0.0277,0.166667
3,15.0,0.010417,0.0727,0.1605,0.0179,0.0243,0.1011,0.0599,0.0044,0.0075,...,0.0005,0.0007,0.0142,0.0020,0.0029,0.0372,0.0293,0.0251,0.0361,0.250000
4,20.0,0.013889,0.0685,0.1656,0.0416,0.0458,0.1104,0.0649,0.0047,0.0133,...,0.0007,0.0008,0.0264,0.0031,0.0030,0.0386,0.0318,0.0285,0.0440,0.333333


### Функции

In [4]:
# Linear BOI and Q
def D(Q):
  x = (Q - 2.7764)/7.9688
  return x

# def D0(Q):
#   # x = (Q - 22.355)/10.93
#   x = (Q + 63.3995)/10.8707
#   return x

# def D0(Q):
#   x = (Q - 22.355)/10.93
#   return x

def D0(Q):
  x = (Q + 31.9236)/8.9363
  return x

def A0(Q):
  x = (Q + 39.478)/9.862
  return x

def A(Q):
  x = (Q + 16.548)/7.9411
  return x

def B0(Q):
  x = (Q + 41.036)/9.9758
  return x

def B(Q):
  x = (Q + 10.56)/7.9659
  return x

def C0(Q):
  x = (Q + 30.168)/8.7912
  return x

def C(Q):
  x = (Q + 18.662)/7.3071
  return x


def renorm(_data, _max, _min):
  return _data*(_max-_min)+_min

def Renormalize(_data, _max, _min):
  renormalize_data = pd.DataFrame()
  for i in range(0, _data.shape[1]-1):
    renormalize_data[_data.columns[i+1]] = renorm(_data.iloc[:, i+1], _max[i], _min[i])
  return renormalize_data


def norm(_data):
    return (_data-_data[1:].min())/(_data.max()-_data[1:].min())



def left_rect(x, f):
  _charge = []
  sum_charge = 0
  h = x[1]-x[0]

  for i in range(0, f.shape[0]):
    sum_charge += h*f[i]
    _charge.append(sum_charge)
  return _charge


def dynamic_function_call(prefix, value):
    """Динамически вызывает функцию по названию префикса."""
    function_name = prefix.strip()
    if function_name in globals():
        return globals()[function_name](value)
    else:
        raise ValueError(f"Функция {function_name} не найдена.")

def BOI(_charge):
    results = []
    for col in _charge.columns:
        prefix = col.split('(')[0]
        values = _charge[col].dropna().values
        if len(values) > 0:
            result = dynamic_function_call(prefix, values[-1])
            results.append(result)
    return results

def percentage_estimate(real_boi, pred_boi):
  _mape = []
  for i in range(0, len(pred_boi)):
    _mape.append(mean_absolute_percentage_error([real_boi[i]], [pred_boi[i]]))
  return _mape

def charge(_x, _data):
  _charge = pd.DataFrame()
  _charge['long'] = pd.Series([i for i in range(1511)])
  for i in range(_data.shape[1]):
    q = left_rect(_x, _data.iloc[:, i].values[:np.count_nonzero(~np.isnan(_data.iloc[:, i].values))]/100)
    _charge[_data.columns[i]] = q + [None] * (1511 - len(q))

  _charge.drop('long', axis=1, inplace=True)
  return _charge

def Normalize(k, _data, _max_index):
  normalize_data = pd.DataFrame()
  normalize_data['long'] = pd.Series([i for i in range(1511)])
  for i in range(_data.shape[1]):
    normalize_data[_data.columns[i]] = norm(_data.iloc[1:k[i], i])

  max_values = []
  min_values = []

  for i in range(_data.shape[1]):
    max_values.append(_data.iloc[_max_index[i], i])
    min_values.append(_data.iloc[1:k[i], i].min())
  normalize_data.drop('long', axis=1, inplace=True)
  normalize_data.drop(normalize_data.index[0], axis=0, inplace=True)

  return (normalize_data, max_values, min_values)

### Берём временные ряды, которые будем аппроксимировать

In [5]:
choice_data = df.iloc[:-1, 2:-1]

choice_data = choice_data.iloc[:, np.r_[:27, 57:66]]
REAL = REAL[np.r_[:27, 57:66]]

In [6]:
def find_indices_multiple_columns(db, initial_points=12, total_points=289):
    results = {}

    for column in db.columns:
        current_index = 12
        indices = []
        initial_points=12
        total_points=289

        while (current_index < len(db)) and (len(indices) < total_points):
            end_index = current_index + initial_points

            if end_index > len(db):
                break

            points = db[column][current_index:end_index]

            derivative = np.diff(points)
            mean_derivative = np.mean(derivative)


            if mean_derivative < 0:
                indices.append(end_index)
                break
            else:
                current_index += 4

            if initial_points > 260:
                indices.append(144)
                break
        if not indices:
            indices.append(144)
        results[column] = indices

    return results

result_indices = find_indices_multiple_columns(choice_data.iloc[:289, :])

`result_indices` - словарь с ключами - названиями колонок, значениями - первыми точками, после которых начинается устойчивое убывание значений (отрицательная средняя производная). Если такая точка не найдена, возвращает индекс 144.

In [7]:
result_indices

{"A0('11-25')": [24],
 "A0('9-16' )": [24],
 "A0('9-10')": [24],
 "A0('9-20')": [24],
 "A0('9-24')": [24],
 "A0('9-27')": [52],
 "A0('9-11')": [160],
 "A0('11-28')": [216],
 "A0('10-03')": [176],
 "B0('11-25')": [24],
 "B0('9-16')": [24],
 "B0('9-10')": [24],
 "B0('9-20')": [24],
 "B0(9-24')": [24],
 "B0('9-27')": [32],
 "B0('9-11')": [184],
 "B0('11-28')": [192],
 "B0('10-03')": [256],
 "C0('11-25')": [24],
 "C0('9-16')": [24],
 "C0('9-10')": [24],
 "C0('9-20')": [24],
 "C0('9-24')": [24],
 "C0('9-27')": [48],
 "C0('9-11')": [152],
 "C0('11-28')": [256],
 "C0('10-03')": [144],
 "D0('11-25')": [24],
 "D0('9-16' )": [28],
 "D0('9-10')": [24],
 "D0('9-20')": [28],
 "D0('9-24')": [32],
 "D0('9-27')": [52],
 "D0('9-11')": [144],
 "D0('11-28')": [120],
 "D0('10-03')": [108]}